# GA4 日指标异常审计

## tl;dr

在 2020-11-01 至 2020-11-14 的公开 GA4 样例中，前 10 天没有规范订单，11 月 11–14 日连续出现规范订单与收入。MAD 检测将这四天的订单/收入相关指标标记为高异常；由于历史窗口很短且中位数为零，这只能作为统计发现和数据质量核验线索，不能当作因果结论或生产告警。

## Context & Methods

### Key Assumptions

- 输入批次为 `9a2d8d7c-0f5b-4c57-a54d-98c59b9c5a27`。
- 规范订单排除 `(not set)` 与 `unknown` 交易号，并按交易号去重。
- 检测使用仅向后看的 7 天窗口、至少 5 个历史点、中位数与 MAD、稳健 Z 阈值 3.5。
- 报告仅使用聚合 Mart 与 Ops 结果，不读取原始用户级明细。

In [1]:
from pathlib import Path
import os

import pandas as pd
import pymysql
from dotenv import load_dotenv

project_root = Path.cwd().resolve()
if not (project_root / '.env').exists():
    project_root = project_root.parent
load_dotenv(project_root / '.env')
batch_id = '9a2d8d7c-0f5b-4c57-a54d-98c59b9c5a27'
anomaly_run_id = '71a6d517-8df8-454e-87a6-35c7e5181046'
connection = pymysql.connect(
    host=os.environ.get('MYSQL_HOST', '127.0.0.1'),
    port=int(os.environ.get('MYSQL_PORT', '3306')),
    user=os.environ.get('MYSQL_USER', 'root'),
    password=os.environ['MYSQL_ROOT_PASSWORD'],
    charset='utf8mb4',
)


## Data

### 1. Load Reviewed Daily Metrics

In [2]:
daily = pd.read_sql(
    """
    SELECT metric_date, observed_sessions, checkout_start_sessions,
           canonical_order_sessions, canonical_orders, canonical_revenue_usd,
           checkout_to_canonical_order_rate
    FROM insightops_mart.ga4_daily_metric
    WHERE source_batch_id = %s AND definition_version = 'ga4_daily_metrics_v1'
    ORDER BY metric_date
    """,
    connection,
    params=(batch_id,),
)
anomalies = pd.read_sql(
    """
    SELECT metric_date, metric_id, metric_value, baseline_points, baseline_median,
           baseline_mad, robust_z_score, status
    FROM insightops_ops.metric_anomaly_result
    WHERE run_id = %s
    ORDER BY metric_date, metric_id
    """,
    connection,
    params=(anomaly_run_id,),
)
daily['metric_date'] = pd.to_datetime(daily['metric_date'])
anomalies['metric_date'] = pd.to_datetime(anomalies['metric_date'])
assert len(daily) == 14
assert len(anomalies) == 14 * 6
daily


/var/folders/vx/ctbyqz_x2g7169l40g5971w40000gn/T/ipykernel_6973/1877002602.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  daily = pd.read_sql(
/var/folders/vx/ctbyqz_x2g7169l40g5971w40000gn/T/ipykernel_6973/1877002602.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  anomalies = pd.read_sql(


,metric_date,observed_sessions,checkout_start_sessions,canonical_order_sessions,canonical_orders,canonical_revenue_usd,checkout_to_canonical_order_rate
0,2020-11-01,2625,95,0,0,0.0,0.000000
1,2020-11-02,3726,130,0,0,0.0,0.000000
2,2020-11-03,5347,144,0,0,0.0,0.000000
3,2020-11-04,4325,118,0,0,0.0,0.000000
4,2020-11-05,3840,127,0,0,0.0,0.000000
5,2020-11-06,3319,154,0,0,0.0,0.000000
6,2020-11-07,2271,111,0,0,0.0,0.000000
7,2020-11-08,2163,100,0,0,0.0,0.000000
8,2020-11-09,3271,135,0,0,0.0,0.000000
9,2020-11-10,3824,173,0,0,0.0,0.000000


## Results

### 2. Inspect the Daily Order and Revenue Pattern

In [3]:
daily[['metric_date', 'observed_sessions', 'checkout_start_sessions', 'canonical_orders', 'canonical_revenue_usd', 'checkout_to_canonical_order_rate']]

high_anomalies = anomalies[anomalies['status'].eq('ANOMALY_HIGH')].copy()
high_anomalies


,metric_date,metric_id,metric_value,baseline_points,baseline_median,baseline_mad,robust_z_score,status
60,2020-11-11,ga4_canonical_order_sessions,28.000000,7,0.0,0.0,NaN,ANOMALY_HIGH
61,2020-11-11,ga4_canonical_orders,28.000000,7,0.0,0.0,NaN,ANOMALY_HIGH
62,2020-11-11,ga4_canonical_revenue_usd,1461.000000,7,0.0,0.0,NaN,ANOMALY_HIGH
64,2020-11-11,ga4_checkout_to_canonical_order_rate,0.171779,7,0.0,0.0,NaN,ANOMALY_HIGH
66,2020-11-12,ga4_canonical_order_sessions,50.000000,7,0.0,0.0,NaN,ANOMALY_HIGH
67,2020-11-12,ga4_canonical_orders,53.000000,7,0.0,0.0,NaN,ANOMALY_HIGH
68,2020-11-12,ga4_canonical_revenue_usd,3287.000000,7,0.0,0.0,NaN,ANOMALY_HIGH
70,2020-11-12,ga4_checkout_to_canonical_order_rate,0.295858,7,0.0,0.0,NaN,ANOMALY_HIGH
72,2020-11-13,ga4_canonical_order_sessions,69.000000,7,0.0,0.0,NaN,ANOMALY_HIGH
73,2020-11-13,ga4_canonical_orders,78.000000,7,0.0,0.0,NaN,ANOMALY_HIGH


### 3. Reconcile Detection Output and Inspect Contributions

In [4]:
reconciliation = anomalies.groupby('metric_id')['status'].value_counts().unstack(fill_value=0)
assert len(high_anomalies) == 16
assert (anomalies['status'].eq('INSUFFICIENT_HISTORY').groupby(anomalies['metric_id']).sum() == 5).all()
reconciliation

contributions = pd.read_sql(
    """
    SELECT r.metric_date, r.metric_id, c.dimension_name, c.dimension_value,
           c.current_value, c.baseline_median, c.contribution_delta,
           c.contribution_share, c.contribution_rank
    FROM insightops_ops.metric_anomaly_contribution c
    JOIN insightops_ops.metric_anomaly_result r ON r.anomaly_id = c.anomaly_id
    WHERE r.run_id = %s
      AND r.metric_date = '2020-11-13'
      AND r.metric_id = 'ga4_canonical_revenue_usd'
    ORDER BY c.dimension_name, c.contribution_rank
    """,
    connection,
    params=(anomaly_run_id,),
)
assert not contributions.empty
contributions.groupby('dimension_name').head(6)


/var/folders/vx/ctbyqz_x2g7169l40g5971w40000gn/T/ipykernel_6973/1003347151.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  contributions = pd.read_sql(


,metric_date,metric_id,dimension_name,dimension_value,current_value,baseline_median,contribution_delta,contribution_share,contribution_rank
0,2020-11-13,ga4_canonical_revenue_usd,acquisition_channel,google / organic,1885.0,0.0,1885.0,0.327826,3
1,2020-11-13,ga4_canonical_revenue_usd,acquisition_channel,(data deleted) / (data deleted),1261.0,0.0,1261.0,0.219304,4
2,2020-11-13,ga4_canonical_revenue_usd,acquisition_channel,(direct) / (none),1066.0,0.0,1066.0,0.185391,5
3,2020-11-13,ga4_canonical_revenue_usd,acquisition_channel,<Other> / <Other>,728.0,0.0,728.0,0.126609,6
4,2020-11-13,ga4_canonical_revenue_usd,acquisition_channel,<Other> / referral,531.0,0.0,531.0,0.092348,7
5,2020-11-13,ga4_canonical_revenue_usd,acquisition_channel,shop.googlemerchandisestore.com / referral,259.0,0.0,259.0,0.045043,8
10,2020-11-13,ga4_canonical_revenue_usd,device_category,desktop,3203.0,0.0,3203.0,0.557043,1
11,2020-11-13,ga4_canonical_revenue_usd,device_category,mobile,2547.0,0.0,2547.0,0.442957,2
12,2020-11-13,ga4_canonical_revenue_usd,device_category,tablet,0.0,0.0,0.0,0.000000,13


## Takeaways

- 84 个日期×指标检测结果全部可复现；每个指标前 5 天因历史不足被明确标记，而不是静默判为正常。
- 16 个高异常全部集中于 11 月 11–14 日的规范订单相关指标；流量与结账启动量未出现异常。
- 11 月 13 日收入构成在设备上为 desktop 55.70%、mobile 44.30%，渠道中 `google / organic` 为 32.78%。这些是同一日相对历史中位数的描述性切片，不是因果归因。
- 如替换为更长的生产时间序列，应重跑检测而非复用当前零基线结论。

In [5]:
connection.close()
